In [9]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

system_message = "You are a helpful personal assistant."

In [10]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_calendar",
            "description": "Check calendar events for a given date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string"
                    }
                },
                "required": ["date"]
            }
        }
    }
]

In [11]:
def check_calendar(date):
    return "10am: Standup, 2pm: Dentist"

In [12]:
def execute_tool(name, args):
    if name == "check_calendar":
        return check_calendar(**args)

    return f"Unknown tool: {name}"

In [13]:
messages = [
    {
        "role": "system",
        "content": system_message
    },
    {
        "role": "user",
        "content": "What's on my calendar today?"
    }
]

In [14]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools
)

In [15]:
print(response.choices[0].finish_reason)
print(response.choices[0].message.tool_calls)

tool_calls
[ChatCompletionMessageToolCall(id='31srcpq1c', function=Function(arguments='{"date":"today"}', name='check_calendar'), type='function')]


In [ ]:
msg = response.choices[0].message

# Add the assistant's tool-call request to conversation history
messages.append(msg)

for tc in msg.tool_calls:
    # Execute the requested tool
    result = execute_tool(
        tc.function.name,
        json.loads(tc.function.arguments)
    )

    # Add the tool's result to conversation history
    messages.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "content": result
    })

# Ask the LLM to generate a final answer using the tool result
final = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools
)

print(final.choices[0].message.content)